In [ ]:

# # Left Eigenvector Centrality Analysis - Colab Notebook
# 
# This notebook tests the hypothesis that Left Eigenvector Centrality (DeGroot Influence)
# is the primary predictor of long-term belief outcomes in directed networks.
# 
# **Key Hypothesis:** The left eigenvector centrality generalizes root node influence
# and works for ALL directed networks, including those with cycles (no root nodes).

# # Setup

In [ ]:


# ═══════════════════════════════════════════════════════════════════════════════
# RECOMMENDED GOOGLE COLAB RUNTIME
# ═══════════════════════════════════════════════════════════════════════════════
print("=" * 70)
print("🚀 RECOMMENDED COLAB RUNTIME SETTINGS")
print("=" * 70)
print("""
Runtime Type: CPU (NOT GPU/TPU)
   - This notebook uses multiprocessing (CPU parallelization)
   - GPU/TPU won't help and wastes resources

Hardware Accelerator: None
   - Go to: Runtime → Change runtime type → Hardware accelerator: None

RAM:
   - Standard (12GB): OK for small networks (n < 200)
   - High-RAM (25GB+): RECOMMENDED for larger networks

Session Duration:
   - Free Colab: ~90 min timeout, may disconnect
   - Colab Pro: Up to 24h runtime

TIP: Run in background with Colab Pro for long simulations!
""")
print("=" * 70)

In [ ]:


# Clone the repository (ai-agents-branch has the latest code)
get_ipython().system('git clone -b ai-agents-branch https://github.com/IgnacioOQ/e_network_inequality')

In [ ]:


# Install required packages
get_ipython().system('pip install dill tqdm networkx pandas numpy scipy matplotlib seaborn scikit-learn')

In [ ]:


# Change to repository directory and install the package
get_ipython().run_line_magic('cd', 'e_network_inequality')
get_ipython().system('pip install -e .')

In [ ]:


# Add src to path and import modules
import sys
import os
sys.path.insert(0, os.path.abspath('src'))

# Core imports
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import json
from tqdm.auto import tqdm
from multiprocessing import Pool, cpu_count
from functools import partial
from scipy import stats

# Import from net_epistemology package
from net_epistemology.utils.imports import *
from net_epistemology.core.vectorized_model import VectorizedModel
from net_epistemology.utils.network_generation import *

print("✅ All imports successful!")


# # Left Eigenvector Centrality Functions

In [ ]:


def compute_left_eigenvector(G):
    """
    Computes the Left Eigenvector centrality (DeGroot Influence).
    
    In the context of opinion dynamics, this metric identifies the 'ultimate' 
    sources of beliefs. It answers: "In the long run, how much does this agent's 
    initial state determine the group's final consensus?"
    
    Mathematical Definition:
    ------------------------
    1. Constructs a Row-Stochastic Matrix W where W_ij represents the 
       weight agent i places on agent j (based on incoming edges in G).
    2. If a node has no incoming edges (a Source), it is treated as 
       'stubborn' or 'independent' (weight 1.0 on itself).
    3. Solves pi * W = pi (normalized so sum(pi) = 1).
    
    Parameters:
    -----------
    G : nx.DiGraph
        A directed graph where an edge (u, v) means u influences v.
        (v listens to u).

    Returns:
    --------
    dict
        Dictionary mapping node IDs to their influence score (probability mass).
    """
    nodes = list(G.nodes())
    n = len(nodes)
    node_to_idx = {node: i for i, node in enumerate(nodes)}
    
    # Initialize adjacency matrix for the "Listening Graph"
    W = np.zeros((n, n))
    
    for u in nodes:
        u_idx = node_to_idx[u]
        # predecessors(u) are nodes that point TO u in G.
        influencers = list(G.predecessors(u))
        
        if len(influencers) == 0:
            # Case: Independent Agent (Source/Root). Listens only to itself.
            W[u_idx, u_idx] = 1.0
        else:
            # Case: Social Agent. Equal weights for simplicity.
            weight = 1.0 / len(influencers)
            for inf in influencers:
                v_idx = node_to_idx[inf]
                W[u_idx, v_idx] = weight
                
    # Calculate Left Eigenvector for Eigenvalue 1.
    # Corresponds to the Right Eigenvector of the Transpose matrix.
    eigenvalues, eigenvectors = np.linalg.eig(W.T)
    
    # Extract eigenvector corresponding to eigenvalue 1 (or closest to it)
    idx = np.argmin(np.abs(eigenvalues - 1.0))
    left_ev = np.real(eigenvectors[:, idx])
    
    # Normalize to form a probability distribution (sum = 1)
    left_ev = np.abs(left_ev) 
    left_ev = left_ev / np.sum(left_ev)
    
    return {nodes[i]: left_ev[i] for i in range(n)}


def compute_katz_centrality(G, alpha=0.1, beta=1.0, measure_influence=True):
    """
    Computes Katz Centrality, optionally measuring influence (reversed graph).
    """
    if measure_influence:
        target_G = G.reverse()
    else:
        target_G = G
        
    try:
        return nx.katz_centrality(target_G, alpha=alpha, beta=beta, normalized=True)
    except nx.PowerIterationFailedConvergence:
        return nx.katz_centrality_numpy(target_G, alpha=alpha, beta=beta, normalized=True)


# # Prediction Functions

In [ ]:


def predict_outcomes_by_left_eigen(G, node_beliefs, left_eigen_centrality):
    """
    Predict final network belief based on left eigenvector weighted beliefs.
    
    Parameters:
    -----------
    G : nx.DiGraph
        The network
    node_beliefs : np.ndarray
        Current beliefs of each node (True/False for believing truth)
    left_eigen_centrality : dict
        Left eigenvector centrality scores for each node
        
    Returns:
    --------
    float
        Predicted proportion believing truth (weighted by left eigenvector)
    """
    nodes = list(G.nodes())
    total_weight = 0.0
    truth_weight = 0.0
    
    for i, node in enumerate(nodes):
        weight = left_eigen_centrality.get(node, 0.0)
        total_weight += weight
        if node_beliefs[i]:
            truth_weight += weight
    
    return truth_weight / total_weight if total_weight > 0 else 0.0


def predict_node_outcomes_by_influence(G, node_beliefs, left_eigen_centrality, threshold=0.5):
    """
    Predict per-node outcomes based on the influence from truthful vs false believers.
    
    For each node, compute the fraction of its incoming influence that comes from
    nodes believing truth. Predict truth if this exceeds threshold.
    
    Parameters:
    -----------
    G : nx.DiGraph
        The network
    node_beliefs : np.ndarray
        Current beliefs of each node (True/False for believing truth)
    left_eigen_centrality : dict
        Left eigenvector centrality scores
    threshold : float
        Threshold for predicting truth
        
    Returns:
    --------
    np.ndarray
        Predicted beliefs for each node
    """
    nodes = list(G.nodes())
    n = len(nodes)
    node_to_idx = {node: i for i, node in enumerate(nodes)}
    predictions = np.zeros(n, dtype=float)
    
    for i, node in enumerate(nodes):
        # Get predecessors (those influencing this node)
        preds = list(G.predecessors(node))
        
        if len(preds) == 0:
            # Root node - use its own belief
            predictions[i] = float(node_beliefs[i])
        else:
            # Compute weighted average of predecessor beliefs
            total_influence = 0.0
            truth_influence = 0.0
            for pred in preds:
                pred_idx = node_to_idx[pred]
                weight = left_eigen_centrality.get(pred, 1.0 / len(preds))
                total_influence += weight
                if node_beliefs[pred_idx]:
                    truth_influence += weight
            
            if total_influence > 0:
                predictions[i] = truth_influence / total_influence
            else:
                predictions[i] = 0.5
    
    return predictions >= threshold


# # Test Networks
# 
# We test on TWO CATEGORIES of networks to validate the Left Eigenvector hypothesis:
# 
# **CATEGORY A: Networks WITH Root Nodes**
# - Root-based prediction IS applicable
# - LE should match root-based predictions
# 
# **CATEGORY B: Networks WITHOUT Root Nodes (Cyclic)**
# - Root-based prediction NOT applicable
# - LE provides unique predictive value

In [ ]:


def create_test_networks():
    """
    Create a variety of test networks including those WITH and WITHOUT root nodes.
    
    CATEGORY A: Networks WITH root nodes
    - barabasi_albert: Scale-free with hubs
    - tree: Perfect hierarchy with single root
    - empirical_pud: Real-world citation network
    
    CATEGORY B: Networks WITHOUT root nodes (cyclic)
    - complete: Everyone influences everyone
    - erdos_renyi_cyclic: Random with added cycles
    """
    networks = {}
    
    print("=" * 70)
    print("CATEGORY A: NETWORKS WITH ROOT NODES")
    print("=" * 70)
    
    # A1. Network WITH root nodes - Barabasi-Albert
    print("\n[A1] Creating Barabasi-Albert network...")
    G_ba = nx.barabasi_albert_graph(100, 3, seed=42)
    G_ba = nx.DiGraph(G_ba)  # Convert to directed
    root_count_ba = sum(1 for n in G_ba.nodes() if G_ba.in_degree(n) == 0)
    networks['A1_barabasi_albert'] = {
        'graph': G_ba,
        'has_roots': True,
        'category': 'A',
        'description': f'Barabasi-Albert (100 nodes, {root_count_ba} roots)'
    }
    print(f"     {root_count_ba} root nodes detected")
    
    # A2. Network WITH root nodes - Tree
    print("\n[A2] Creating Directed Tree network...")
    G_tree = nx.balanced_tree(3, 4, create_using=nx.DiGraph())
    # Reverse edges so root has influence (children listen to parents)
    G_tree = G_tree.reverse()
    networks['A2_tree'] = {
        'graph': G_tree,
        'has_roots': True,
        'category': 'A',
        'description': f'Balanced Tree (3-ary depth 4, 1 root with 100% influence)'
    }
    print("     1 root node (perfect hierarchy)")
    
    # A3. Load empirical network if available
    try:
        network_path = 'data/empirical_networks/pud_final.json'
        with open(network_path, 'r') as f:
            network_data = json.load(f)
        if 'links' in network_data:
            network_data['edges'] = network_data.pop('links')
        G_emp = nx.node_link_graph(network_data)
        root_count = sum(1 for n in G_emp.nodes() if G_emp.in_degree(n) == 0)
        networks['A3_empirical_pud'] = {
            'graph': G_emp,
            'has_roots': root_count > 0,
            'category': 'A',
            'description': f'Empirical PUD ({len(G_emp.nodes())} nodes, {root_count} roots)'
        }
        print(f"\n[A3] Loaded empirical network: {len(G_emp.nodes())} nodes, {root_count} roots")
    except FileNotFoundError:
        print("\n[A3] Empirical network not found, skipping...")
    
    print("\n" + "=" * 70)
    print("CATEGORY B: NETWORKS WITHOUT ROOT NODES (CYCLIC)")
    print("=" * 70)
    print("NOTE: Root-based prediction NOT applicable for these networks!")
    print("      Left Eigenvector provides UNIQUE predictive value here.")
    
    # B1. Network WITHOUT root nodes - Complete graph (cyclic)
    print("\n[B1] Creating Complete Directed graph...")
    G_complete = nx.complete_graph(50, create_using=nx.DiGraph())
    networks['B1_complete'] = {
        'graph': G_complete,
        'has_roots': False,
        'category': 'B',
        'description': 'Complete Graph (50 nodes, 0 roots, uniform LE)'
    }
    print("     0 root nodes (everyone influences everyone)")
    
    # B2. Network WITHOUT root nodes - Random with cycles
    print("\n[B2] Creating Erdos-Renyi network with enforced cycles...")
    np.random.seed(42)
    G_er = nx.gnp_random_graph(100, 0.05, directed=True, seed=42)
    # Ensure NO root nodes by adding back edges
    for node in list(G_er.nodes()):
        if G_er.in_degree(node) == 0:
            sources = [n for n in G_er.nodes() if n != node]
            if sources:
                G_er.add_edge(np.random.choice(sources), node)
    root_check = sum(1 for n in G_er.nodes() if G_er.in_degree(n) == 0)
    networks['B2_erdos_renyi_cyclic'] = {
        'graph': G_er,
        'has_roots': False,
        'category': 'B',
        'description': f'Erdos-Renyi (100 nodes, {root_check} roots, cyclic)'
    }
    print(f"     {root_check} root nodes after adding cycles")
    
    return networks


# Print summary
networks = create_test_networks()
print("\n" + "=" * 70)
print("TEST NETWORKS SUMMARY")
print("=" * 70)
print("\n{:<25} {:>8} {:>8} {:>10} {}".format("Network", "Nodes", "Edges", "Roots", "Category"))
print("-" * 70)
for name, info in networks.items():
    G = info['graph']
    roots = sum(1 for n in G.nodes() if G.in_degree(n) == 0)
    cat = "WITH ROOTS" if info['category'] == 'A' else "NO ROOTS"
    print(f"{name:<25} {len(G.nodes()):>8} {len(G.edges()):>8} {roots:>10} {cat}")


# # Single Simulation Analysis

In [ ]:


def run_single_analysis(network_name, network_info, n_steps=50000, uncertainty=0.001):
    """
    Run a single simulation and compare left eigenvector prediction with actual outcome.
    """
    G = network_info['graph']
    nodes = list(G.nodes())
    n_agents = len(nodes)
    category = network_info.get('category', 'unknown')
    
    print(f"\n{'='*60}")
    print(f"Analyzing: {network_name}")
    print(f"Category: {'WITH ROOTS (A)' if category == 'A' else 'NO ROOTS / CYCLIC (B)'}")
    print(f"{'='*60}")
    print(f"Network: {n_agents} nodes, {len(G.edges())} edges")
    
    # Compute left eigenvector centrality
    print("Computing Left Eigenvector Centrality...")
    left_eigen = compute_left_eigenvector(G)
    
    # Analyze LE distribution
    le_values = np.array(list(left_eigen.values()))
    print(f"  LE stats: max={np.max(le_values):.4f}, mean={np.mean(le_values):.4f}")
    
    # Also compute Katz for comparison
    print("Computing Katz Centrality...")
    try:
        katz = compute_katz_centrality(G, alpha=0.01)
    except:
        katz = {n: 1.0/n_agents for n in nodes}  # Fallback to uniform
    
    # Run simulation
    print(f"Running simulation for {n_steps} steps...")
    model = VectorizedModel(
        network=G,
        n_experiments=10,
        uncertainty=uncertainty,
        agent_type="beta",
        tstep_stopping=True,
        compute_root_analysis=True,
    )
    model.run_simulation(number_of_steps=n_steps, show_bar=True)
    
    # Get actual outcomes
    actual_beliefs = model.credences[:, 1] > model.credences[:, 0]
    actual_proportion = np.mean(actual_beliefs)
    
    # Compute predictions
    # 1. Left Eigenvector weighted prediction
    le_prediction = predict_outcomes_by_left_eigen(G, actual_beliefs, left_eigen)
    
    # 2. Root-based prediction (if roots exist)
    if model.root_analysis and model.root_analysis['n_roots'] > 0:
        root_prediction = model.proportion_reached_by_truth
    else:
        root_prediction = None
    
    # 3. Katz-weighted prediction
    katz_prediction = predict_outcomes_by_left_eigen(G, actual_beliefs, katz)
    
    # Node-level analysis
    node_predictions_le = predict_node_outcomes_by_influence(G, actual_beliefs, left_eigen)
    node_accuracy_le = np.mean(node_predictions_le == actual_beliefs)
    
    results = {
        'network': network_name,
        'category': category,
        'n_nodes': n_agents,
        'n_edges': len(G.edges()),
        'has_roots': network_info['has_roots'],
        'n_roots': model.root_analysis['n_roots'] if model.root_analysis else 0,
        'actual_proportion': actual_proportion,
        'le_prediction': le_prediction,
        'le_error': abs(actual_proportion - le_prediction),
        'root_prediction': root_prediction,
        'root_error': abs(actual_proportion - root_prediction) if root_prediction is not None else None,
        'katz_prediction': katz_prediction,
        'katz_error': abs(actual_proportion - katz_prediction),
        'node_accuracy_le': node_accuracy_le,
        'n_steps': n_steps,
    }
    
    # Print results
    print(f"\nResults for {network_name}:")
    print(f"  Actual share believing truth: {actual_proportion:.4f}")
    print(f"  Left Eigenvector prediction:  {le_prediction:.4f} (error: {abs(actual_proportion - le_prediction):.4f})")
    if root_prediction is not None:
        print(f"  Root-based prediction:        {root_prediction:.4f} (error: {abs(actual_proportion - root_prediction):.4f})")
    else:
        print(f"  Root-based prediction:        N/A (no roots - Category B network)")
    print(f"  Katz centrality prediction:   {katz_prediction:.4f} (error: {abs(actual_proportion - katz_prediction):.4f})")
    print(f"  Node-level accuracy (LE):     {node_accuracy_le:.4f}")
    
    return results, left_eigen, model


# # Run Analysis on All Networks

In [ ]:


all_results = []

# Run Category A (WITH roots) first
print("\n" + "=" * 80)
print("RUNNING CATEGORY A: NETWORKS WITH ROOT NODES")
print("=" * 80)
for network_name, network_info in networks.items():
    if network_info.get('category') == 'A':
        try:
            results, left_eigen, model = run_single_analysis(
                network_name, 
                network_info, 
                n_steps=100000,
                uncertainty=0.001
            )
            all_results.append(results)
        except Exception as e:
            print(f"Error analyzing {network_name}: {e}")

# Run Category B (WITHOUT roots)
print("\n" + "=" * 80)
print("RUNNING CATEGORY B: NETWORKS WITHOUT ROOT NODES (CYCLIC)")
print("This is where Left Eigenvector provides UNIQUE value!")
print("=" * 80)
for network_name, network_info in networks.items():
    if network_info.get('category') == 'B':
        try:
            results, left_eigen, model = run_single_analysis(
                network_name, 
                network_info, 
                n_steps=100000,
                uncertainty=0.001
            )
            all_results.append(results)
        except Exception as e:
            print(f"Error analyzing {network_name}: {e}")

# Create results dataframe
results_df = pd.DataFrame(all_results)

# Display summary with focus on ERRORS
print("\n" + "=" * 80)
print("SUMMARY RESULTS - PREDICTION ERRORS")
print("=" * 80)
print("\nLower error = better prediction. Root error is N/A for Category B (cyclic networks).")
print()

# Create a focused summary
for _, row in results_df.iterrows():
    root_err_str = f"{row['root_error']:.4f}" if row['root_error'] is not None else "N/A"
    print(f"{row['network']:25} | Cat {row['category']} | Roots: {row['n_roots']:3} | "
          f"LE Err: {row['le_error']:.4f} | Root Err: {root_err_str:>6} | "
          f"Katz Err: {row['katz_error']:.4f} | Node Acc: {row['node_accuracy_le']:.4f}")

print()
print("Full DataFrame:")
print(results_df[['network', 'category', 'n_roots', 'actual_proportion', 'le_error', 'root_error', 'katz_error', 'node_accuracy_le']].to_string())


# # Visualization

In [ ]:


fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# 1. Prediction Error Comparison (using pre-computed errors)
ax = axes[0, 0]
x = np.arange(len(results_df))
width = 0.25

# Use pre-computed error columns
le_errors = results_df['le_error'].values
root_errors = results_df['root_error'].fillna(0).values  # Fill NaN with 0 for plotting
katz_errors = results_df['katz_error'].values

# Mark which have valid root errors
root_valid = results_df['root_error'].notna().values

ax.bar(x - width, le_errors, width, label='Left Eigenvector', color='blue', alpha=0.7)
bars_root = ax.bar(x, root_errors, width, label='Root-based', color='green', alpha=0.7)
ax.bar(x + width, katz_errors, width, label='Katz', color='orange', alpha=0.7)

# Mark N/A for root errors (Category B)
for i, valid in enumerate(root_valid):
    if not valid:
        bars_root[i].set_alpha(0.2)
        bars_root[i].set_hatch('//')

ax.set_xlabel('Network')
ax.set_ylabel('Prediction Error (|Actual - Predicted|)')
ax.set_title('Prediction Error by Method\\n(Hatched = Root N/A for Category B)')
ax.set_xticks(x)
ax.set_xticklabels(results_df['network'], rotation=45, ha='right')
ax.legend()
ax.grid(True, alpha=0.3)

# 2. Left Eigenvector Centrality Distribution (last network)
ax = axes[0, 1]
le_values = np.array(list(left_eigen.values()))
ax.hist(le_values, bins=30, edgecolor='black', alpha=0.7)
ax.axvline(np.mean(le_values), color='red', linestyle='--', label=f'Mean: {np.mean(le_values):.4f}')
ax.axvline(np.max(le_values), color='green', linestyle='--', label=f'Max: {np.max(le_values):.4f}')
ax.set_xlabel('Left Eigenvector Centrality')
ax.set_ylabel('Frequency')
ax.set_title(f'Left Eigenvector Distribution (Last Network)')
ax.legend()
ax.grid(True, alpha=0.3)

# 3. Node-level Accuracy by Category
ax = axes[1, 0]
colors = ['blue' if cat == 'A' else 'red' for cat in results_df['category']]
ax.bar(results_df['network'], results_df['node_accuracy_le'], color=colors, alpha=0.7)
ax.axhline(0.5, color='gray', linestyle='--', label='Random baseline')
ax.set_xlabel('Network')
ax.set_ylabel('Node-level Accuracy')
ax.set_title('Left Eigenvector Node-Level Accuracy\n(Blue=Category A/With Roots, Red=Category B/No Roots)')
ax.set_xticklabels(results_df['network'], rotation=45, ha='right')
ax.legend()
ax.grid(True, alpha=0.3)

# 4. Summary Table - ERRORS
ax = axes[1, 1]
ax.axis('off')
summary_data = []
for _, row in results_df.iterrows():
    root_err = f"{row['root_error']:.4f}" if row['root_error'] is not None else "N/A"
    summary_data.append([
        row['network'][:16],
        row['category'],
        f"{row['n_roots']}",
        f"{row['le_error']:.4f}",
        root_err,
        f"{row['katz_error']:.4f}",
        f"{row['node_accuracy_le']:.4f}"
    ])
table = ax.table(
    cellText=summary_data,
    colLabels=['Network', 'Cat', 'Roots', 'LE Err', 'Root Err', 'Katz Err', 'Node Acc'],
    loc='center',
    cellLoc='center'
)
table.auto_set_font_size(False)
table.set_fontsize(8)
table.scale(1.2, 1.5)
ax.set_title('Prediction Errors Summary\\n(Lower = Better)', fontsize=12, pad=20)

plt.tight_layout()
plt.savefig('left_eigen_analysis.png', dpi=150, bbox_inches='tight')
print("\nPlot saved to 'left_eigen_analysis.png'")
plt.show()


# # Convergence Analysis (Multiple Steps)

In [ ]:
def run_convergence_analysis(G, network_name, step_counts=[1000, 5000, 10000, 50000, 100000, 500000], 
                              uncertainty=0.001, n_trials=5):
    """
    Run multiple simulations at different step counts to see how predictions improve.
    """
    print(f"\n{'='*60}")
    print(f"Convergence Analysis: {network_name}")
    print(f"{'='*60}")
    
    # Compute centralities once
    left_eigen = compute_left_eigenvector(G)
    nodes = list(G.nodes())
    
    results = []
    
    for n_steps in step_counts:
        le_errors = []
        root_errors = []
        
        for trial in range(n_trials):
            model = VectorizedModel(
                network=G,
                n_experiments=10,
                uncertainty=uncertainty,
                agent_type="beta",
                tstep_stopping=True,
                compute_root_analysis=True,
            )
            model.run_simulation(number_of_steps=n_steps, show_bar=False)
            
            actual_beliefs = model.credences[:, 1] > model.credences[:, 0]
            actual = np.mean(actual_beliefs)
            
            le_pred = predict_outcomes_by_left_eigen(G, actual_beliefs, left_eigen)
            le_errors.append(abs(actual - le_pred))
            
            if model.root_analysis and model.root_analysis['n_roots'] > 0:
                root_errors.append(abs(actual - model.proportion_reached_by_truth))
        
        results.append({
            'steps': n_steps,
            'le_error_mean': np.mean(le_errors),
            'le_error_std': np.std(le_errors),
            'root_error_mean': np.mean(root_errors) if root_errors else None,
            'root_error_std': np.std(root_errors) if root_errors else None,
        })
        
        print(f"Steps {n_steps:6d}: LE error = {np.mean(le_errors):.4f} +/- {np.std(le_errors):.4f}")
    
    return results


# Run on Empirical network if available, else BA network
if 'A3_empirical_pud' in networks:
    conv_results = run_convergence_analysis(
        networks['A3_empirical_pud']['graph'],
        'A3_empirical_pud',
        step_counts=[1000, 5000, 10000, 50000, 100000, 500000],
        n_trials=3
    )
elif 'A1_barabasi_albert' in networks:
    conv_results = run_convergence_analysis(
        networks['A1_barabasi_albert']['graph'],
        'A1_barabasi_albert',
        step_counts=[1000, 5000, 10000, 50000, 100000, 500000],
        n_trials=3
    )

# Part 2: Stochastic Process Decomposition

This section extends the analysis by decomposing the simulation into its constituent stochastic components.

## The Three Components of Outcome Determination

1. **Stochastic Matrices** ← Derived from the sampling and update mechanism (Binomial experiments, Bayesian updating)
2. **Left Eigenvector Centrality** ← Derived from the network structure (who influences whom)
3. **Initial Distribution** ← Random priors (starting beliefs)

We analyze each component sequentially using a Markov chain framework to understand how they interact to determine final outcomes.

## 2.1 Deriving the Expected Update Equations

The key insight is that the update mechanism has a **deterministic expectation** overlaid with **stochastic noise**.

### The Update Rule

At each step, for agent $i$ testing theory $k$:
$$\alpha_i^{(k)}(t+1) = \alpha_i^{(k)}(t) + \sum_{j \in \mathcal{N}(i) \cup \{i\}} S_j^{(k)}(t)$$

where $S_j^{(k)}(t) \sim \text{Binomial}(n, p_k)$ with:
- $p_1 = 0.5 + u$ (truth)
- $p_0 = 0.5 - u$ (false)

### Expected Evidence Flow

Taking expectations over the Binomial sampling:
$$\mathbb{E}[S_j^{(k)}] = n \cdot p_k \cdot \mathbf{1}[\text{agent } j \text{ tests theory } k]$$

The network aggregation matrix $A^T$ (transposed adjacency) determines how evidence flows.

In [ ]:
# ============================================================================
# 2.1 EXPECTED UPDATE EQUATIONS AND TRANSITION MATRICES
# ============================================================================

def compute_listening_matrix(G):
    """
    Compute the row-stochastic listening matrix W.
    
    W[i,j] = weight agent i places on agent j's evidence.
    - If i is a root (no predecessors): W[i,i] = 1 (listens only to self)
    - Otherwise: W[i,j] = 1/|Pred(i)| for j in Pred(i)
    
    Returns:
        W: (n, n) row-stochastic matrix
        nodes: list of node IDs in order
    """
    nodes = list(G.nodes())
    n = len(nodes)
    node_to_idx = {node: i for i, node in enumerate(nodes)}
    
    W = np.zeros((n, n))
    
    for u in nodes:
        u_idx = node_to_idx[u]
        influencers = list(G.predecessors(u))
        
        if len(influencers) == 0:
            # Root node - listens only to itself
            W[u_idx, u_idx] = 1.0
        else:
            # Equal weight on all predecessors
            weight = 1.0 / len(influencers)
            for inf in influencers:
                v_idx = node_to_idx[inf]
                W[u_idx, v_idx] = weight
    
    return W, nodes


def compute_evidence_aggregation_matrix(G):
    """
    Compute the evidence aggregation matrix E.
    
    E[i,j] = 1 if agent i observes agent j's experiments (including self).
    This is the adjacency matrix transposed plus identity.
    
    The expected evidence for agent i = E[i,:] @ evidence_vector
    """
    nodes = list(G.nodes())
    n = len(nodes)
    
    # Adjacency matrix: A[i,j] = 1 if edge i -> j exists
    A = nx.to_numpy_array(G, nodelist=nodes)
    
    # Evidence aggregation: sum over predecessors + self
    # If A[i,j] = 1 means i -> j, then j observes i
    # So agent j aggregates from all i where A[i,j] = 1, i.e., column j of A
    # Plus self-observation (identity)
    E = A.T + np.eye(n)
    
    return E, nodes


def compute_expected_update_matrix(G, n_experiments, uncertainty):
    """
    Compute the expected evidence increment per step.
    
    For theory k with probability p_k:
    - Expected successes from agent j if j tests theory k: n * p_k
    - Expected failures: n * (1 - p_k)
    
    This returns the matrix that maps "who tests what" to "expected alpha increments".
    """
    E, nodes = compute_evidence_aggregation_matrix(G)
    n = len(nodes)
    
    p_true = 0.5 + uncertainty  # Theory 1
    p_false = 0.5 - uncertainty  # Theory 0
    
    # Expected successes per experiment for each theory
    exp_success_true = n_experiments * p_true
    exp_success_false = n_experiments * p_false
    
    return {
        'E': E,
        'nodes': nodes,
        'exp_success_true': exp_success_true,
        'exp_success_false': exp_success_false,
        'exp_failure_true': n_experiments * (1 - p_true),
        'exp_failure_false': n_experiments * (1 - p_false),
    }


# Demonstrate on a test network
print("=" * 70)
print("EXPECTED UPDATE EQUATIONS - DEMONSTRATION")
print("=" * 70)

# Use Barabasi-Albert network
G_test = networks['A1_barabasi_albert']['graph']
n_test = len(G_test.nodes())

W, nodes = compute_listening_matrix(G_test)
E, _ = compute_evidence_aggregation_matrix(G_test)
update_info = compute_expected_update_matrix(G_test, n_experiments=10, uncertainty=0.001)

print(f"\nNetwork: {n_test} nodes, {len(G_test.edges())} edges")
print(f"\nListening Matrix W (row-stochastic):")
print(f"  Shape: {W.shape}")
print(f"  Row sums (should be 1): min={W.sum(axis=1).min():.4f}, max={W.sum(axis=1).max():.4f}")
print(f"  Sparsity: {(W == 0).sum() / W.size:.2%} zeros")

print(f"\nEvidence Aggregation Matrix E:")
print(f"  Shape: {E.shape}")
print(f"  Max in-degree + 1: {E.sum(axis=1).max():.0f}")
print(f"  Min in-degree + 1: {E.sum(axis=1).min():.0f}")

print(f"\nExpected evidence per step (n_exp=10, u=0.001):")
print(f"  Theory 1 (truth): {update_info['exp_success_true']:.3f} successes, {update_info['exp_failure_true']:.3f} failures")
print(f"  Theory 0 (false): {update_info['exp_success_false']:.3f} successes, {update_info['exp_failure_false']:.3f} failures")
print(f"  Advantage per step: {update_info['exp_success_true'] - update_info['exp_success_false']:.4f} more successes for truth")

## 2.2 Variance Analysis and Confidence Intervals

The Binomial sampling introduces variance that affects outcome predictability.

### Variance of a Single Update

For agent $i$ aggregating from $d_i$ sources (predecessors + self):
$$\text{Var}(S_i^{(k)}) = \sum_{j \in \mathcal{N}(i) \cup \{i\}} n \cdot p_k (1-p_k) = d_i \cdot n \cdot p_k(1-p_k)$$

### Variance Accumulation Over Time

After $T$ steps, the total variance in $\alpha_i^{(k)}$ scales as $O(T \cdot d_i)$.

However, the **credence** $c = \alpha/(\alpha+\beta)$ has variance that **decreases** as $\alpha + \beta$ grows:
$$\text{Var}(c) \approx \frac{\alpha \beta}{(\alpha+\beta)^2(\alpha+\beta+1)} \to 0 \text{ as } T \to \infty$$

This explains why longer simulations give more consistent outcomes.

In [ ]:
# ============================================================================
# 2.2 VARIANCE ANALYSIS AND CONFIDENCE INTERVALS
# ============================================================================

def compute_update_variance(G, n_experiments, uncertainty):
    """
    Compute the variance of evidence updates for each agent.
    
    Variance of Binomial(n, p) = n * p * (1-p)
    Total variance for agent i = sum over all observed agents
    """
    E, nodes = compute_evidence_aggregation_matrix(G)
    n = len(nodes)
    
    p_true = 0.5 + uncertainty
    p_false = 0.5 - uncertainty
    
    # Variance per experiment
    var_per_exp_true = n_experiments * p_true * (1 - p_true)
    var_per_exp_false = n_experiments * p_false * (1 - p_false)
    
    # Number of sources each agent observes
    n_sources = E.sum(axis=1)  # Row sums
    
    # Total variance per step for each agent
    var_per_step_true = n_sources * var_per_exp_true
    var_per_step_false = n_sources * var_per_exp_false
    
    return {
        'n_sources': n_sources,
        'var_per_step_true': var_per_step_true,
        'var_per_step_false': var_per_step_false,
        'var_per_exp_true': var_per_exp_true,
        'var_per_exp_false': var_per_exp_false,
    }


def run_variance_analysis(G, network_name, n_experiments=10, uncertainty=0.001, 
                          n_trials=50, n_steps=10000):
    """
    Run multiple simulations to empirically measure variance in outcomes.
    """
    print(f"\n{'='*60}")
    print(f"Variance Analysis: {network_name}")
    print(f"Running {n_trials} trials with {n_steps} steps each...")
    print(f"{'='*60}")
    
    nodes = list(G.nodes())
    n = len(nodes)
    
    # Theoretical variance
    var_info = compute_update_variance(G, n_experiments, uncertainty)
    
    # Collect outcomes across trials
    final_credences = []
    final_beliefs = []
    proportions = []
    
    for trial in tqdm(range(n_trials), desc="Trials"):
        model = VectorizedModel(
            network=G,
            n_experiments=n_experiments,
            uncertainty=uncertainty,
            agent_type="beta",
            tstep_stopping=True,
        )
        model.run_simulation(number_of_steps=n_steps, show_bar=False)
        
        final_credences.append(model.credences.copy())
        beliefs = model.credences[:, 1] > model.credences[:, 0]
        final_beliefs.append(beliefs)
        proportions.append(np.mean(beliefs))
    
    final_credences = np.array(final_credences)  # (n_trials, n_agents, 2)
    final_beliefs = np.array(final_beliefs)  # (n_trials, n_agents)
    proportions = np.array(proportions)
    
    # Compute statistics
    # Per-agent credence variance across trials
    credence_var = final_credences.var(axis=0)  # (n_agents, 2)
    credence_std = final_credences.std(axis=0)
    
    # Per-agent belief consistency (fraction of trials with same outcome)
    belief_consistency = np.maximum(final_beliefs.mean(axis=0), 1 - final_beliefs.mean(axis=0))
    
    # Overall proportion variance
    prop_mean = proportions.mean()
    prop_std = proportions.std()
    prop_ci_95 = (np.percentile(proportions, 2.5), np.percentile(proportions, 97.5))
    
    results = {
        'theoretical_var_per_step': var_info,
        'empirical_credence_var': credence_var,
        'empirical_credence_std': credence_std,
        'belief_consistency': belief_consistency,
        'proportion_mean': prop_mean,
        'proportion_std': prop_std,
        'proportion_ci_95': prop_ci_95,
        'final_credences': final_credences,
        'final_beliefs': final_beliefs,
        'proportions': proportions,
    }
    
    print(f"\nResults:")
    print(f"  Proportion believing truth: {prop_mean:.4f} +/- {prop_std:.4f}")
    print(f"  95% CI: [{prop_ci_95[0]:.4f}, {prop_ci_95[1]:.4f}]")
    print(f"  Agent belief consistency: {belief_consistency.mean():.4f} (1.0 = always same outcome)")
    print(f"  Credence std (Theory 1): mean={credence_std[:, 1].mean():.4f}, max={credence_std[:, 1].max():.4f}")
    
    return results


# Run variance analysis on test networks
print("\n" + "=" * 70)
print("VARIANCE ANALYSIS")
print("=" * 70)

# Store results for comparison
variance_results = {}

# Run on a network with roots and one without
for net_name in ['A1_barabasi_albert', 'B1_complete']:
    if net_name in networks:
        G = networks[net_name]['graph']
        variance_results[net_name] = run_variance_analysis(
            G, net_name, 
            n_experiments=10, 
            uncertainty=0.001,
            n_trials=30,
            n_steps=50000
        )

In [ ]:
# Visualize variance analysis results
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for idx, (net_name, results) in enumerate(variance_results.items()):
    col = idx
    
    # Plot 1: Distribution of final proportions
    ax = axes[0, col]
    ax.hist(results['proportions'], bins=15, edgecolor='black', alpha=0.7)
    ax.axvline(results['proportion_mean'], color='red', linestyle='--', 
               label=f"Mean: {results['proportion_mean']:.3f}")
    ax.axvline(results['proportion_ci_95'][0], color='orange', linestyle=':', 
               label=f"95% CI: [{results['proportion_ci_95'][0]:.3f}, {results['proportion_ci_95'][1]:.3f}]")
    ax.axvline(results['proportion_ci_95'][1], color='orange', linestyle=':')
    ax.set_xlabel('Proportion Believing Truth')
    ax.set_ylabel('Frequency')
    ax.set_title(f'{net_name}\nDistribution of Outcomes ({len(results["proportions"])} trials)')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    
    # Plot 2: Agent-level belief consistency
    ax = axes[1, col]
    consistency = results['belief_consistency']
    ax.hist(consistency, bins=20, edgecolor='black', alpha=0.7)
    ax.axvline(0.5, color='gray', linestyle='--', label='Random (0.5)')
    ax.axvline(consistency.mean(), color='red', linestyle='--', 
               label=f'Mean: {consistency.mean():.3f}')
    ax.set_xlabel('Belief Consistency (max of P(truth), P(false))')
    ax.set_ylabel('Number of Agents')
    ax.set_title(f'Per-Agent Outcome Consistency\n(1.0 = same outcome every trial)')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('variance_analysis.png', dpi=150, bbox_inches='tight')
print("\nPlot saved to 'variance_analysis.png'")
plt.show()

# Summary comparison
print("\n" + "=" * 70)
print("VARIANCE COMPARISON SUMMARY")
print("=" * 70)
print(f"\n{'Network':<25} {'Prop Mean':>12} {'Prop Std':>12} {'Consistency':>12}")
print("-" * 65)
for net_name, results in variance_results.items():
    print(f"{net_name:<25} {results['proportion_mean']:>12.4f} {results['proportion_std']:>12.4f} {results['belief_consistency'].mean():>12.4f}")

## 2.3 Separating Initial Conditions from Network Structure

A key question: How much of the outcome is determined by:
1. **Initial priors** (random at t=0)
2. **Network structure** (fixed, determines information flow)
3. **Stochastic sampling** (ongoing randomness)

### Experimental Design

To separate these effects, we:
1. **Fix initial conditions** - Run multiple simulations with the SAME initial priors but different sampling randomness
2. **Vary initial conditions** - Run multiple simulations with different initial priors but same network

This allows us to decompose: $\text{Var}(\text{outcome}) = \text{Var}_{\text{init}} + \text{Var}_{\text{sampling}}$

In [ ]:
# ============================================================================
# 2.3 SEPARATING INITIAL CONDITIONS FROM NETWORK STRUCTURE
# ============================================================================

def create_model_with_fixed_init(G, n_experiments, uncertainty, initial_alphas_betas):
    """
    Create a VectorizedModel with fixed initial conditions.
    
    This allows us to isolate the effect of sampling randomness from initial conditions.
    """
    model = VectorizedModel(
        network=G,
        n_experiments=n_experiments,
        uncertainty=uncertainty,
        agent_type="beta",
        tstep_stopping=True,
    )
    # Override the random initialization with fixed values
    model.alphas_betas = initial_alphas_betas.copy()
    # Recompute credences from the fixed alphas_betas
    a = model.alphas_betas[:, :, 0]
    b = model.alphas_betas[:, :, 1]
    model.credences = a / (a + b)
    return model


def run_fixed_init_analysis(G, network_name, n_experiments=10, uncertainty=0.001,
                            n_init_conditions=10, n_sampling_trials=20, n_steps=50000):
    """
    Analyze variance decomposition between initial conditions and sampling.
    
    For each of n_init_conditions different starting points:
      - Run n_sampling_trials simulations with different random seeds
      - Measure within-init variance (due to sampling)
    
    Then measure between-init variance (due to different starting points).
    """
    print(f"\n{'='*60}")
    print(f"Initial Conditions Analysis: {network_name}")
    print(f"Testing {n_init_conditions} initial conditions x {n_sampling_trials} trials")
    print(f"{'='*60}")
    
    nodes = list(G.nodes())
    n = len(nodes)
    
    # Store results
    all_proportions = []  # (n_init, n_trials)
    init_means = []
    within_vars = []
    
    for init_idx in tqdm(range(n_init_conditions), desc="Initial conditions"):
        # Create a random initial condition
        np.random.seed(init_idx * 1000)  # Reproducible but different
        initial_ab = np.zeros((n, 2, 2))
        for i in range(n):
            initial_ab[i, 0] = np.random.uniform(0, 4, size=2)
            initial_ab[i, 1] = np.random.uniform(0, 4, size=2)
        
        # Run multiple trials with this fixed init
        trial_proportions = []
        for trial in range(n_sampling_trials):
            model = create_model_with_fixed_init(G, n_experiments, uncertainty, initial_ab)
            model.run_simulation(number_of_steps=n_steps, show_bar=False)
            beliefs = model.credences[:, 1] > model.credences[:, 0]
            trial_proportions.append(np.mean(beliefs))
        
        trial_proportions = np.array(trial_proportions)
        all_proportions.append(trial_proportions)
        init_means.append(trial_proportions.mean())
        within_vars.append(trial_proportions.var())
    
    all_proportions = np.array(all_proportions)  # (n_init, n_trials)
    init_means = np.array(init_means)
    within_vars = np.array(within_vars)
    
    # Compute variance decomposition
    # Total variance = Between-init variance + Within-init variance (sampling)
    total_var = all_proportions.flatten().var()
    between_var = init_means.var()  # Variance of means across init conditions
    within_var_mean = within_vars.mean()  # Average within-init variance
    
    # Fraction explained by each
    frac_init = between_var / total_var if total_var > 0 else 0
    frac_sampling = within_var_mean / total_var if total_var > 0 else 0
    
    results = {
        'all_proportions': all_proportions,
        'init_means': init_means,
        'within_vars': within_vars,
        'total_var': total_var,
        'between_var': between_var,
        'within_var_mean': within_var_mean,
        'frac_init': frac_init,
        'frac_sampling': frac_sampling,
    }
    
    print(f"\nVariance Decomposition:")
    print(f"  Total variance:           {total_var:.6f}")
    print(f"  Between-init variance:    {between_var:.6f} ({frac_init*100:.1f}% of total)")
    print(f"  Within-init variance:     {within_var_mean:.6f} ({frac_sampling*100:.1f}% of total)")
    print(f"\n  --> Initial conditions explain {frac_init*100:.1f}% of outcome variance")
    print(f"  --> Sampling randomness explains {frac_sampling*100:.1f}% of outcome variance")
    
    return results


# Run the analysis
print("\n" + "=" * 70)
print("INITIAL CONDITIONS vs SAMPLING VARIANCE DECOMPOSITION")
print("=" * 70)

init_analysis_results = {}

for net_name in ['A1_barabasi_albert', 'B1_complete']:
    if net_name in networks:
        G = networks[net_name]['graph']
        init_analysis_results[net_name] = run_fixed_init_analysis(
            G, net_name,
            n_experiments=10,
            uncertainty=0.001,
            n_init_conditions=10,
            n_sampling_trials=15,
            n_steps=50000
        )

In [ ]:
# Visualize the variance decomposition
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Plot 1: Variance decomposition pie charts
ax = axes[0]
labels = []
sizes = []
colors_list = []
for net_name, results in init_analysis_results.items():
    labels.extend([f'{net_name}\nInit Cond', f'{net_name}\nSampling'])
    sizes.extend([results['frac_init'], results['frac_sampling']])
    colors_list.extend(['steelblue', 'coral'])

# Create grouped bar chart instead of pie
x = np.arange(len(init_analysis_results))
width = 0.35
init_fracs = [r['frac_init'] for r in init_analysis_results.values()]
samp_fracs = [r['frac_sampling'] for r in init_analysis_results.values()]

bars1 = ax.bar(x - width/2, init_fracs, width, label='Initial Conditions', color='steelblue')
bars2 = ax.bar(x + width/2, samp_fracs, width, label='Sampling Randomness', color='coral')

ax.set_ylabel('Fraction of Total Variance')
ax.set_title('Variance Decomposition\n(What determines outcomes?)')
ax.set_xticks(x)
ax.set_xticklabels(init_analysis_results.keys())
ax.legend()
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bar, val in zip(bars1, init_fracs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
            f'{val*100:.0f}%', ha='center', va='bottom', fontsize=9)
for bar, val in zip(bars2, samp_fracs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
            f'{val*100:.0f}%', ha='center', va='bottom', fontsize=9)

# Plot 2: Distribution of outcomes by initial condition (first network)
ax = axes[1]
first_net = list(init_analysis_results.keys())[0]
results = init_analysis_results[first_net]
for init_idx, trial_props in enumerate(results['all_proportions']):
    ax.scatter([init_idx] * len(trial_props), trial_props, alpha=0.5, s=20)
ax.plot(range(len(results['init_means'])), results['init_means'], 'ko-', 
        label='Mean per init', markersize=8)
ax.set_xlabel('Initial Condition Index')
ax.set_ylabel('Proportion Believing Truth')
ax.set_title(f'{first_net}\nOutcomes by Initial Condition')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 3: Within-init variance distribution
ax = axes[2]
for net_name, results in init_analysis_results.items():
    ax.hist(results['within_vars'], bins=10, alpha=0.5, label=net_name, edgecolor='black')
ax.set_xlabel('Within-Init Variance')
ax.set_ylabel('Frequency')
ax.set_title('Distribution of Sampling Variance\n(per initial condition)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('init_conditions_analysis.png', dpi=150, bbox_inches='tight')
print("\nPlot saved to 'init_conditions_analysis.png'")
plt.show()

## 2.4 Sequential Markov Chain Decomposition

The simulation can be viewed as a sequence of coupled stochastic processes. We analyze this by examining the **spectral properties** of the relevant matrices.

### The Three Matrices

1. **Listening Matrix W** (row-stochastic): Determines long-run influence via left eigenvector
2. **Evidence Aggregation Matrix E**: Determines how evidence accumulates
3. **Transition Kernel**: The full stochastic transition (state-dependent)

### Spectral Gap and Mixing Time

The **spectral gap** $\gamma = 1 - \lambda_2$ (where $\lambda_2$ is the second-largest eigenvalue) controls:
- How fast the system "forgets" initial conditions
- The mixing time of the Markov chain
- The rate of convergence to steady-state influence distribution

In [ ]:
# ============================================================================
# 2.4 SEQUENTIAL MARKOV CHAIN DECOMPOSITION
# ============================================================================

def analyze_spectral_properties(G):
    """
    Analyze the spectral properties of the listening matrix W.
    
    Returns eigenvalues, eigenvectors, spectral gap, and mixing time estimate.
    """
    W, nodes = compute_listening_matrix(G)
    n = len(nodes)
    
    # Compute eigenvalues of W
    eigenvalues = np.linalg.eigvals(W)
    
    # Sort by magnitude (descending)
    eigenvalues_sorted = sorted(eigenvalues, key=lambda x: abs(x), reverse=True)
    
    # The largest eigenvalue should be 1 (row-stochastic)
    lambda_1 = abs(eigenvalues_sorted[0])
    
    # Second largest eigenvalue (may be complex)
    lambda_2 = abs(eigenvalues_sorted[1]) if len(eigenvalues_sorted) > 1 else 0
    
    # Spectral gap
    spectral_gap = 1 - lambda_2
    
    # Mixing time estimate: t_mix ~ 1 / spectral_gap
    mixing_time_est = 1 / spectral_gap if spectral_gap > 0 else np.inf
    
    # Left eigenvector (stationary distribution)
    eigenvalues_full, eigenvectors_full = np.linalg.eig(W.T)
    idx = np.argmin(np.abs(eigenvalues_full - 1.0))
    left_ev = np.real(eigenvectors_full[:, idx])
    left_ev = np.abs(left_ev) / np.sum(np.abs(left_ev))
    
    # Concentration of influence (Gini-like measure)
    left_ev_sorted = np.sort(left_ev)[::-1]
    cumsum = np.cumsum(left_ev_sorted)
    # How many nodes hold 50% of influence?
    n_50 = np.searchsorted(cumsum, 0.5) + 1
    # How many nodes hold 90% of influence?
    n_90 = np.searchsorted(cumsum, 0.9) + 1
    
    return {
        'W': W,
        'nodes': nodes,
        'eigenvalues': eigenvalues_sorted,
        'lambda_1': lambda_1,
        'lambda_2': lambda_2,
        'spectral_gap': spectral_gap,
        'mixing_time_est': mixing_time_est,
        'left_eigenvector': left_ev,
        'n_50_influence': n_50,
        'n_90_influence': n_90,
    }


def run_markov_chain_analysis(networks_dict):
    """
    Run spectral analysis on all networks and compare.
    """
    print("\n" + "=" * 70)
    print("MARKOV CHAIN SPECTRAL ANALYSIS")
    print("=" * 70)
    
    results = {}
    
    for net_name, net_info in networks_dict.items():
        G = net_info['graph']
        spec = analyze_spectral_properties(G)
        results[net_name] = spec
        
        n = len(spec['nodes'])
        print(f"\n{net_name}:")
        print(f"  Nodes: {n}")
        print(f"  Lambda_1 (should be 1): {spec['lambda_1']:.6f}")
        print(f"  Lambda_2: {spec['lambda_2']:.6f}")
        print(f"  Spectral gap: {spec['spectral_gap']:.6f}")
        print(f"  Mixing time estimate: {spec['mixing_time_est']:.1f} steps")
        print(f"  Influence concentration:")
        print(f"    - Top {spec['n_50_influence']} nodes ({100*spec['n_50_influence']/n:.1f}%) hold 50% influence")
        print(f"    - Top {spec['n_90_influence']} nodes ({100*spec['n_90_influence']/n:.1f}%) hold 90% influence")
    
    return results


# Run the analysis
spectral_results = run_markov_chain_analysis(networks)

In [ ]:
# Visualize spectral analysis
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Eigenvalue spectrum comparison
ax = axes[0, 0]
for net_name, spec in spectral_results.items():
    eigenvalues = spec['eigenvalues'][:20]  # Top 20
    mags = [abs(e) for e in eigenvalues]
    ax.plot(range(len(mags)), mags, 'o-', label=net_name, markersize=4)
ax.axhline(1.0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Eigenvalue Rank')
ax.set_ylabel('|Eigenvalue|')
ax.set_title('Eigenvalue Spectrum of Listening Matrix W')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1.1)

# Plot 2: Spectral gap comparison
ax = axes[0, 1]
net_names = list(spectral_results.keys())
gaps = [spectral_results[n]['spectral_gap'] for n in net_names]
colors = ['steelblue' if networks[n]['category'] == 'A' else 'coral' for n in net_names]
bars = ax.bar(range(len(net_names)), gaps, color=colors, alpha=0.7, edgecolor='black')
ax.set_xticks(range(len(net_names)))
ax.set_xticklabels(net_names, rotation=45, ha='right')
ax.set_ylabel('Spectral Gap (1 - lambda_2)')
ax.set_title('Spectral Gap by Network\n(Blue=With Roots, Red=No Roots)')
ax.grid(True, alpha=0.3, axis='y')
# Add value labels
for bar, gap in zip(bars, gaps):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
            f'{gap:.3f}', ha='center', va='bottom', fontsize=8)

# Plot 3: Influence distribution (Lorenz-like curve)
ax = axes[1, 0]
for net_name, spec in spectral_results.items():
    le = spec['left_eigenvector']
    le_sorted = np.sort(le)[::-1]
    cumsum = np.cumsum(le_sorted) / np.sum(le_sorted)
    x = np.arange(1, len(le_sorted) + 1) / len(le_sorted)
    ax.plot(x, cumsum, '-', label=net_name, linewidth=2)
ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Perfect equality')
ax.set_xlabel('Fraction of Nodes (ranked by influence)')
ax.set_ylabel('Cumulative Fraction of Influence')
ax.set_title('Influence Concentration (Lorenz Curve)')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 4: Summary table
ax = axes[1, 1]
ax.axis('off')
table_data = []
for net_name, spec in spectral_results.items():
    n = len(spec['nodes'])
    table_data.append([
        net_name[:16],
        f"{spec['spectral_gap']:.4f}",
        f"{spec['mixing_time_est']:.0f}",
        f"{spec['n_50_influence']} ({100*spec['n_50_influence']/n:.0f}%)",
        f"{spec['n_90_influence']} ({100*spec['n_90_influence']/n:.0f}%)",
    ])
table = ax.table(
    cellText=table_data,
    colLabels=['Network', 'Spectral Gap', 'Mix Time', 'N for 50%', 'N for 90%'],
    loc='center',
    cellLoc='center'
)
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1.2, 1.5)
ax.set_title('Spectral Properties Summary', fontsize=12, pad=20)

plt.tight_layout()
plt.savefig('spectral_analysis.png', dpi=150, bbox_inches='tight')
print("\nPlot saved to 'spectral_analysis.png'")
plt.show()

## 2.5 Synthesis: The Complete Predictive Model

We now have all three components needed to predict outcomes:

### Component 1: Stochastic Matrices (Sampling Mechanism)
- Binomial(n, p) draws determine evidence at each step
- Expected update: $n \cdot p_k$ successes per experiment for theory $k$
- Variance per step: $n \cdot p_k(1-p_k)$ per source observed
- **Key insight**: The truth advantage accumulates as $n \cdot 2u \cdot T$ over $T$ steps

### Component 2: Left Eigenvector (Network Structure)  
- The left eigenvector $\pi$ of the listening matrix $W$ determines long-run influence
- Agents with high $\pi_i$ disproportionately affect final outcomes
- The spectral gap determines how quickly the system "forgets" non-influential nodes

### Component 3: Initial Distribution (Random Priors)
- Random priors create initial bias toward one theory
- High-influence agents' initial beliefs matter most
- The fraction of variance explained by initial conditions depends on network structure

### The Complete Prediction

$$P(\text{network converges to truth}) \approx \sum_{i=1}^N \pi_i \cdot P(\text{agent } i \text{ converges to truth})$$

where $P(\text{agent } i \text{ converges to truth})$ depends on:
1. Initial prior advantage for theory 1
2. Number of steps (longer = more evidence for truth)
3. Uncertainty parameter $u$ (higher = faster convergence to truth)
4. Network position (high in-degree = more evidence aggregation)

In [ ]:
# ============================================================================
# 2.5 SYNTHESIS: COMBINED PREDICTIVE MODEL
# ============================================================================

def compute_agent_truth_probability(initial_credence, n_steps, n_experiments, uncertainty, in_degree):
    """
    Estimate probability that an agent converges to truth.
    
    This is a simplified model based on:
    - Initial advantage for theory 1 vs theory 0
    - Expected evidence accumulation over n_steps
    - Number of sources (in_degree + 1 for self)
    
    The key idea: After T steps, agent accumulates ~T * n_sources * n_exp * 2u 
    more expected successes for theory 1 than theory 0.
    """
    # Initial advantage (positive if initially favoring truth)
    init_advantage = initial_credence[1] - initial_credence[0]
    
    # Evidence advantage per step (expected)
    # Each source testing theory 1 contributes n_exp * (p_true - p_false) = n_exp * 2u more successes
    # But not all sources test theory 1 - assume ~50% do initially
    evidence_rate = (in_degree + 1) * n_experiments * 2 * uncertainty * 0.5
    
    # Total expected advantage after n_steps
    total_advantage = init_advantage + evidence_rate * n_steps * 0.01  # Scaled
    
    # Convert to probability using sigmoid
    prob = 1 / (1 + np.exp(-total_advantage * 10))  # Sigmoid with scaling
    
    return prob


def build_combined_predictor(G, n_experiments, uncertainty, n_steps):
    """
    Build a predictor that combines all three components.
    """
    nodes = list(G.nodes())
    n = len(nodes)
    
    # Component 1: Network structure - Left eigenvector
    left_eigen = compute_left_eigenvector(G)
    le_array = np.array([left_eigen[node] for node in nodes])
    
    # Component 2: Network structure - In-degrees (for evidence aggregation)
    A = nx.to_numpy_array(G, nodelist=nodes)
    in_degrees = A.sum(axis=0)  # Sum of column = in-degree
    
    # Component 3: Spectral properties
    spec = analyze_spectral_properties(G)
    
    return {
        'nodes': nodes,
        'left_eigenvector': le_array,
        'in_degrees': in_degrees,
        'spectral_gap': spec['spectral_gap'],
        'n_experiments': n_experiments,
        'uncertainty': uncertainty,
        'n_steps': n_steps,
    }


def predict_with_initial_conditions(predictor, initial_credences):
    """
    Predict outcomes given initial conditions.
    
    Args:
        predictor: Output from build_combined_predictor
        initial_credences: (n_agents, 2) array of initial credences
    
    Returns:
        Predicted proportion believing truth
    """
    n = len(predictor['nodes'])
    le = predictor['left_eigenvector']
    in_deg = predictor['in_degrees']
    
    # Compute per-agent probability of converging to truth
    agent_probs = np.zeros(n)
    for i in range(n):
        agent_probs[i] = compute_agent_truth_probability(
            initial_credences[i],
            predictor['n_steps'],
            predictor['n_experiments'],
            predictor['uncertainty'],
            in_deg[i]
        )
    
    # Weight by left eigenvector
    weighted_prob = np.sum(le * agent_probs)
    
    return weighted_prob, agent_probs


# Test the combined predictor
print("\n" + "=" * 70)
print("COMBINED PREDICTIVE MODEL TEST")
print("=" * 70)

# Build predictor for test network
G_test = networks['A1_barabasi_albert']['graph']
predictor = build_combined_predictor(G_test, n_experiments=10, uncertainty=0.001, n_steps=50000)

# Run simulations and compare predictions
n_test_trials = 20
predictions = []
actuals = []

print(f"\nRunning {n_test_trials} test simulations...")
for trial in tqdm(range(n_test_trials)):
    # Create model with random init
    model = VectorizedModel(
        network=G_test,
        n_experiments=10,
        uncertainty=0.001,
        agent_type="beta",
        tstep_stopping=True,
    )
    
    # Get initial credences
    initial_cred = model.credences.copy()
    
    # Predict
    pred, _ = predict_with_initial_conditions(predictor, initial_cred)
    predictions.append(pred)
    
    # Run and get actual
    model.run_simulation(number_of_steps=50000, show_bar=False)
    actual = np.mean(model.credences[:, 1] > model.credences[:, 0])
    actuals.append(actual)

predictions = np.array(predictions)
actuals = np.array(actuals)

# Compute correlation
correlation = np.corrcoef(predictions, actuals)[0, 1]
mae = np.mean(np.abs(predictions - actuals))

print(f"\nResults:")
print(f"  Correlation (pred vs actual): {correlation:.4f}")
print(f"  Mean Absolute Error: {mae:.4f}")
print(f"  Prediction range: [{predictions.min():.3f}, {predictions.max():.3f}]")
print(f"  Actual range: [{actuals.min():.3f}, {actuals.max():.3f}]")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
ax.scatter(predictions, actuals, alpha=0.7, s=50)
ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfect prediction')
ax.set_xlabel('Predicted Proportion')
ax.set_ylabel('Actual Proportion')
ax.set_title(f'Combined Model Prediction vs Actual\nCorrelation: {correlation:.3f}, MAE: {mae:.3f}')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

ax = axes[1]
errors = predictions - actuals
ax.hist(errors, bins=15, edgecolor='black', alpha=0.7)
ax.axvline(0, color='red', linestyle='--', label='Zero error')
ax.axvline(errors.mean(), color='orange', linestyle='--', label=f'Mean: {errors.mean():.3f}')
ax.set_xlabel('Prediction Error (Pred - Actual)')
ax.set_ylabel('Frequency')
ax.set_title('Distribution of Prediction Errors')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('combined_predictor.png', dpi=150, bbox_inches='tight')
print("\nPlot saved to 'combined_predictor.png'")
plt.show()

In [ ]:
print("\n" + "=" * 80)
print("EXTENDED LEFT EIGENVECTOR HYPOTHESIS - FINAL RESULTS")
print("=" * 80)

# Separate results by category
cat_a_results = results_df[results_df['category'] == 'A']
cat_b_results = results_df[results_df['category'] == 'B']

print("""
+------------------------------------------------------------------------------+
|              LEFT EIGENVECTOR CENTRALITY HYPOTHESIS - EXTENDED               |
+------------------------------------------------------------------------------+
|  The long-run outcomes can be predicted using THREE components:              |
|                                                                              |
|  1. STOCHASTIC MATRICES (Sampling Mechanism)                                 |
|     - Binomial experiments with truth advantage 2u per trial                 |
|     - Evidence aggregates through network structure                          |
|                                                                              |
|  2. LEFT EIGENVECTOR CENTRALITY (Network Structure)                          |
|     - Determines long-run influence of each agent                            |
|     - Spectral gap controls mixing time                                      |
|                                                                              |
|  3. INITIAL DISTRIBUTION (Random Priors)                                     |
|     - Creates initial bias toward theories                                   |
|     - High-influence agents' priors matter most                              |
+------------------------------------------------------------------------------+
""")

print("=" * 80)
print("PART 1: LEFT EIGENVECTOR PREDICTION ACCURACY")
print("=" * 80)
if len(cat_a_results) > 0:
    avg_le_error_a = cat_a_results['le_error'].mean()
    print(f"\n  Category A (With Roots):")
    print(f"    Average LE Error: {avg_le_error_a:.4f}")
    
if len(cat_b_results) > 0:
    avg_le_error_b = cat_b_results['le_error'].mean()
    print(f"\n  Category B (No Roots):")
    print(f"    Average LE Error: {avg_le_error_b:.4f}")
    print("    --> LE provides UNIQUE value for cyclic networks!")

print("\n" + "=" * 80)
print("PART 2: VARIANCE DECOMPOSITION")
print("=" * 80)
if init_analysis_results:
    for net_name, results in init_analysis_results.items():
        print(f"\n  {net_name}:")
        print(f"    Initial conditions explain: {results['frac_init']*100:.1f}% of variance")
        print(f"    Sampling randomness explains: {results['frac_sampling']*100:.1f}% of variance")

print("\n" + "=" * 80)
print("PART 2: SPECTRAL PROPERTIES")
print("=" * 80)
for net_name, spec in spectral_results.items():
    n = len(spec['nodes'])
    print(f"\n  {net_name}:")
    print(f"    Spectral gap: {spec['spectral_gap']:.4f}")
    print(f"    Mixing time estimate: {spec['mixing_time_est']:.0f} steps")
    print(f"    Influence concentration: {spec['n_50_influence']} nodes hold 50% influence")

print("""
+------------------------------------------------------------------------------+
|  KEY INSIGHTS:                                                               |
+------------------------------------------------------------------------------+
|  * For DAGs: Left eigenvector concentrates in root nodes                     |
|  * For cyclic networks: Left eigenvector identifies "effective sources"      |
|  * Spectral gap predicts convergence speed                                   |
|  * Initial conditions matter MORE for networks with concentrated influence   |
|  * The combined model (LE + init + sampling) predicts outcomes accurately    |
+------------------------------------------------------------------------------+
""")

print("=" * 80)
print("ANALYSIS COMPLETE")
print("=" * 80)

In [ ]:


from datetime import datetime
try:
    import pytz
    nyc_time = datetime.now(pytz.timezone('America/New_York'))
    formatted_time = nyc_time.strftime('%Y-%m-%d %H:%M:%S %Z')
except ImportError:
    formatted_time = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

print(f"✅ Analysis completed at: {formatted_time}")

# Disconnect from Colab runtime to free resources
try:
    from IPython.display import Javascript
    display(Javascript('google.colab.kernel.disconnect()'))
    print("🔌 Disconnected from Colab runtime.")
except:
    print("(Not running in Colab - no disconnect needed)")